# 03 - Regression Analysis
**Syllabus link:** Simple Linear Regression, Multiple Linear Regression, Polynomial Regression, OLS estimation, correlations, MAE / MSE / RMSE / R2

**Goal:** predict the transaction `Amount` (a continuous target) from the PCA features `V1-V28`.
This demonstrates the regression techniques on the same dataset. Low R2 is expected, because the V-features were built by PCA to describe transaction behaviour, not to predict the amount. This is reported honestly.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# khel khtm

sns.set_style("whitegrid")
FIG_DIR = "../reports/figures"
os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(f"{FIG_DIR}/{name}.png", dpi=150, bbox_inches="tight")

df = pd.read_csv("../data/processed/creditcard_clean.csv")
v_cols = [f"V{i}" for i in range(1, 29)]
X = df[v_cols]
y = df["Amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 1. Correlation of features with `Amount`

In [ ]:
corr_amt = df[v_cols + ["Amount"]].corr()["Amount"].drop("Amount")
corr_amt = corr_amt.reindex(corr_amt.abs().sort_values(ascending=False).index)
print(corr_amt.head(8).round(3))

plt.figure(figsize=(8, 7))
corr_amt.sort_values().plot(kind="barh")
plt.title("Correlation of V-features with Amount")
savefig("03_corr_with_amount"); plt.show()

best_feat = corr_amt.index[0]
top5 = corr_amt.index[:5].tolist()
print("Best single feature:", best_feat, "| Top 5:", top5)

In [ ]:
def reg_metrics(name, model, Xtr, Xte):
    pred_tr, pred_te = model.predict(Xtr), model.predict(Xte)
    mse = mean_squared_error(y_test, pred_te)
    return {
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred_te),
        "MSE": mse,
        "RMSE": np.sqrt(mse),
        "R2 (test)": r2_score(y_test, pred_te),
        "R2 (train)": r2_score(y_train, pred_tr),
    }

results = []

## 2. Simple linear regression (one feature)

In [ ]:
slr = LinearRegression().fit(X_train[[best_feat]], y_train)
print(f"Amount = {slr.intercept_:.3f} + ({slr.coef_[0]:.3f}) * {best_feat}")
results.append(reg_metrics(f"Simple Linear ({best_feat})", slr, X_train[[best_feat]], X_test[[best_feat]]))

plt.figure(figsize=(7, 5))
sample = X_test[[best_feat]].join(y_test).sample(3000, random_state=1)
plt.scatter(sample[best_feat], sample["Amount"], s=8, alpha=0.4)
xs = np.linspace(sample[best_feat].min(), sample[best_feat].max(), 100)
plt.plot(xs, slr.predict(pd.DataFrame({best_feat: xs})), color="red")
plt.xlabel(best_feat); plt.ylabel("Amount"); plt.title("Simple linear regression fit")
savefig("03_simple_linear"); plt.show()

## 3. Multiple linear regression (all V-features)

In [ ]:
mlr = LinearRegression().fit(X_train, y_train)
results.append(reg_metrics("Multiple Linear (V1-V28)", mlr, X_train, X_test))

coef = pd.Series(mlr.coef_, index=v_cols).sort_values(key=abs, ascending=False)
print("Intercept:", round(mlr.intercept_, 3))
coef.head(10).round(3)

## 4. Polynomial regression

In [ ]:
X5_train, X5_test = X_train[top5], X_test[top5]

lin5 = LinearRegression().fit(X5_train, y_train)
results.append(reg_metrics("Linear (top 5 features)", lin5, X5_train, X5_test))

for deg in [2, 3]:
    poly = make_pipeline(PolynomialFeatures(degree=deg, include_bias=False), LinearRegression())
    poly.fit(X5_train, y_train)
    results.append(reg_metrics(f"Polynomial deg {deg} (top 5)", poly, X5_train, X5_test))

Degree 1 on the same five features is the fair baseline for polynomial terms. If train R2 rises but test R2 does not, the model is overfitting.

## 5. OLS estimation (statsmodels)

In [ ]:
X_sm = sm.add_constant(X_train)
ols = sm.OLS(y_train, X_sm).fit()
print(ols.summary())

In [ ]:
ols_tab = pd.DataFrame({"coef": ols.params, "std err": ols.bse, "t": ols.tvalues, "p-value": ols.pvalues})
ols_tab["significant (p<0.05)"] = ols_tab["p-value"] < 0.05
print("R2:", round(ols.rsquared, 4), "| Adj R2:", round(ols.rsquared_adj, 4), "| F p-value:", ols.f_pvalue)
ols_tab[ols_tab["significant (p<0.05)"]].sort_values("p-value").round(4)

**How to read the OLS output (viva points):**
- `coef`: change in Amount for a one-unit change in that feature, other features fixed.
- `p-value < 0.05`: coefficient is statistically different from zero.
- `R-squared`: share of the variance in Amount explained by the model.
- `Adj. R-squared`: R2 penalised for the number of predictors.
- OLS minimises the sum of squared residuals.

## 6. Metrics comparison

In [ ]:
res = pd.DataFrame(results).set_index("Model").round(4)
res

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
res["RMSE"].plot(kind="barh", ax=ax[0]); ax[0].set_title("RMSE (lower is better)")
res["R2 (test)"].plot(kind="barh", ax=ax[1], color="seagreen"); ax[1].set_title("R2 on test set (higher is better)")
plt.tight_layout(); savefig("03_regression_metrics"); plt.show()

## 7. Residual diagnostics (multiple linear regression)

In [ ]:
pred = mlr.predict(X_test)
resid = y_test - pred

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
idx = np.random.RandomState(0).choice(len(y_test), 4000, replace=False)
ax[0].scatter(np.asarray(y_test)[idx], pred[idx], s=6, alpha=0.4)
ax[0].plot([0, y_test.max()], [0, y_test.max()], "r--")
ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted"); ax[0].set_title("Actual vs predicted")
ax[1].scatter(pred[idx], np.asarray(resid)[idx], s=6, alpha=0.4)
ax[1].axhline(0, color="r", ls="--")
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("Residual"); ax[1].set_title("Residuals vs predicted")
ax[2].hist(resid, bins=80); ax[2].set_title("Residual distribution")
plt.tight_layout(); savefig("03_residuals"); plt.show()

## Interpretation (copy into the report)
- R2 values are low: the PCA features explain only a small part of the variation in Amount.
- Residuals are not normal and show non-constant spread (heteroscedasticity) because Amount is highly skewed. This violates OLS assumptions, so coefficient p-values should be read with caution.
- Polynomial terms change the fit little compared with the matching linear model, so extra complexity does not pay off here.
- Regression is a learning exercise on this dataset; the project's main task is classification (notebook 04).

Next: `04_classification_models.ipynb`.